In [0]:
import requests
import time
from datetime import date
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType

API_KEY = "your_api_key_here"
SYMBOLS = ["AAPL", "GOOGL", "AMZN", "MSFT"]

schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("open", StringType(), True),
    StructField("high", StringType(), True),
    StructField("low", StringType(), True),
    StructField("price", StringType(), True),
    StructField("volume", StringType(), True),
    StructField("latest_trading_day", StringType(), True),
    StructField("previous_close", StringType(), True),
    StructField("change", StringType(), True),
    StructField("change_pct", StringType(), True),
    StructField("ingested_date", StringType(), True)
])

rows = []

for symbol in SYMBOLS:
    try:
        url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&outputsize=compact&apikey={API_KEY}"
        response = requests.get(url, timeout=10)
        data = response.json().get("Time Series (Daily)", {})

        if not data:
            print(f"WARNING: No historical data for {symbol}")
            continue

        for trade_date, values in data.items():
            rows.append(Row(
                symbol=symbol,
                open=values.get("1. open"),
                high=values.get("2. high"),
                low=values.get("3. low"),
                price=values.get("4. close"),
                volume=values.get("5. volume"),
                latest_trading_day=trade_date,
                previous_close=None,
                change=None,
                change_pct=None,
                ingested_date=str(date.today())
            ))

        print(f"SUCCESS: {symbol} — {len(data)} days of history fetched")

    except Exception as e:
        print(f"ERROR: {symbol} — {e}")

    time.sleep(15)

backfill_df = spark.createDataFrame(rows, schema=schema)

(
    backfill_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("my_catalog.default.bronze_stock_quotes_backfill")
)

print(f"Backfill complete — {len(rows)} total rows written")

SUCCESS: AAPL — 100 days of history fetched
SUCCESS: GOOGL — 100 days of history fetched
SUCCESS: AMZN — 100 days of history fetched
SUCCESS: MSFT — 100 days of history fetched
Backfill complete — 400 total rows written
